# Romanian Satire Detection · Nitro NLP 2024

**Nitro Language Processing Hackathon, 3rd edition** · 23-hour Kaggle competition · 30–31 March 2024 · team *Multiplying Tensors* · 🏅 **Honorable Mention**

Given a Romanian news article (title + body), predict whether it is **satire (1)** or **real news (0)**. Submissions are scored with **balanced accuracy**, the average of the recall on each class, so a model gains nothing by favouring the more frequent class.

This notebook is the pipeline we built during the hackathon, split into sections and documented afterwards. The preprocessing and modelling code is the code we ran; the few post-hackathon changes are listed at the end.

| Step | What happens | Section |
|---|---|---|
| Text cleaning | strip URLs, HTML tags and numbers; normalise Romanian diacritics, quotes and dashes | 3 |
| Tokenisation | title + body → a Romanian-specific tokenizer (Romanian BERT, RoBERT or Romanian mT5), 64 tokens | 3 |
| Model | pretrained transformer encoder + custom fully-connected classification head | 5 |
| Training | class-weighted cross-entropy, AdamW, 3 epochs | 6–8 |
| Inference | predict the test set and write `submission.csv` | 9 |

**Running it:** in a Kaggle notebook, attach the [competition data](https://www.kaggle.com/competitions/nitro-language-processing-3/data), switch on a GPU and run all cells (training takes about 20 minutes). The README explains how to run it locally.

## 1. Setup

During the hackathon the code lived in a few Python modules (preprocessing, dataset, models, training loop, utilities) that we pasted into a single Kaggle cell. Each module now has its own section.

The code needs **Python 3.10+** because it uses `X | Y` type hints.

In [ ]:
import random
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Dict, Generic, List, Tuple, TypeVar

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pandas import DataFrame
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score)
from torch import Tensor, optim
from torch.optim import Optimizer
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, BatchEncoding, BertModel, MT5EncoderModel

In [ ]:
# Folder with train.csv and test.csv. With the competition data attached to a Kaggle notebook this is
# /kaggle/input/nitro-language-processing-3/ (during the event we read the files from our own Kaggle
# dataset, /kaggle/input/new-nitronlp2/).
DATA_DIR = Path('/kaggle/input/nitro-language-processing-3/')

# Encoder to fine-tune: 'bert' (Romanian BERT), 'RoBERT' or 'mt5'. The run reported in section 10 used 'bert'.
MODEL_NAME = 'bert'

SEED = 61
DEVICE: torch.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')


class Args(object):
    """Training hyper-parameters, shared by all encoders."""

    def __init__(self) -> None:
        self.num_epochs = 3
        self.num_workers = 1
        self.batch_size = 64
        self.weight_decay = 5e-6
        self.learning_rate = 2e-5

## 2. Data

The competition provides `train.csv` and `test.csv`. Each row is an article with a `title` and a `content` field. The content is sometimes a single paragraph rather than the whole article, and either field can be empty. Training rows also carry the label in `class` (`True` = satire).

We concatenate title and body into a single `text` field, so the model sees the headline and the opening of the article together, and map the labels to `1` (satire) / `0` (real news).

In [ ]:
def read_data(data_dir: Path, train_filename: str = 'train.csv', test_filename: str = 'test.csv') -> (
        Tuple[pd.DataFrame, pd.DataFrame]):
    train_data_raw = pd.read_csv(data_dir / train_filename, sep=',', index_col=None, encoding='utf-8')
    test_data_raw = pd.read_csv(data_dir / test_filename, sep=',', index_col=None, encoding='utf-8')
    train_data_raw['class'] = train_data_raw['class'].map({True: 1, False: 0})

    # Title + body -> one text field (a missing part becomes an empty string)
    train_data_raw['text'] = train_data_raw['title'].fillna('') + ' ' + train_data_raw['content'].fillna('')
    test_data_raw['text'] = test_data_raw['title'].fillna('') + ' ' + test_data_raw['content'].fillna('')

    # The submission uses each test row's position as its id (section 9)
    train_data_raw = train_data_raw.drop(columns='id', errors='ignore')
    test_data_raw = test_data_raw.drop(columns='id', errors='ignore')

    return train_data_raw, test_data_raw


def compute_metrics(labels, preds):
    """Full metric report for a labelled split, e.g. a hold-out validation set."""
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    balanced_accuracy = (tp / (tp + fn) + specificity) / 2
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy,
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds),
        'specificity': specificity
    }

In [ ]:
train_data_frame, test_data_frame = read_data(DATA_DIR, 'train.csv', 'test.csv')

print(f'train: {len(train_data_frame):,} articles, test: {len(test_data_frame):,} articles')
train_data_frame['class'].value_counts(normalize=True).rename({0: 'real news', 1: 'satire'})

## 3. Text cleaning and tokenisation

Articles scraped from news sites are noisy, so every text goes through an automated cleaner before tokenisation:

| Step | Why |
|---|---|
| drop texts that contain a tab character | a guard against malformed rows |
| remove URLs, HTML tags and numbers (regex) | boilerplate that says little about satire and uses up the 64-token budget |
| collapse repeated whitespace | |
| `ş ţ Ş Ţ` → `ș ț Ș Ț` | Romanian text mixes the legacy *cedilla* letters with the correct *comma-below* ones. Tokenizers treat them as different characters, so the same word could be split into different tokens |
| typographic quotes `„ “ ”` → `"`, remove soft hyphens and dashes | the same word or quote always reaches the tokenizer in the same form |
| lowercase (RoBERT only) | RoBERT is an uncased model |

We also experimented with removing punctuation, e-mail addresses and a list of click-bait words (`ȘOC`, `BREAKING NEWS`, `EXCLUSIV`, ...) so that the model could not rely on them. Those steps are commented out because they were not part of the final pipeline.

The cleaned text is then tokenised with the tokenizer of the encoder being fine-tuned and padded or truncated to **64 tokens**. `AutoPreprocessor` works with any Hugging Face tokenizer, and the subclasses pin the three Romanian encoders we tried:

| Preprocessor | Hugging Face checkpoint | Notes |
|---|---|---|
| `BERTPreprocessor` | [`dumitrescustefan/bert-base-romanian-cased-v1`](https://huggingface.co/dumitrescustefan/bert-base-romanian-cased-v1) | Romanian BERT, cased |
| `RoBERTPreprocessor` | [`readerbench/RoBERT-base`](https://huggingface.co/readerbench/RoBERT-base) | RoBERT, uncased |
| `MT5Preprocessor` | [`dumitrescustefan/mt5-base-romanian`](https://huggingface.co/dumitrescustefan/mt5-base-romanian) | Romanian mT5 (SentencePiece) |

In [ ]:
# Click-bait / sensationalist markers. Removing them (BAIT_WORDS_REGEX) was one of the experiments that
# did not make it into the final pipeline.
BAIT_WORDS = [
    "ALERT(Ă|A)", "alerta", "Alertă",
    "ANALIZA", "analiza", "Analiza",
    "Ancheta", "ANCHETA", "ancheta",
    "BREAKING NEWS", "Breaking News", "breaking news",
    "CAPTURA", "captura", "Captura",
    "CLIP", "clip", "Clip",
    "CONFLICT", "conflict", "Conflict",
    "CONFRUNTARE", "confruntare", "Confruntare",
    "CONTROVERSAT", "controversat", "Controversat",
    "CRIZ(Ă|A)", "criză", "Criză",
    "DECLARAȚIE", "declarație", "Declarație",
    "DEZASTRU", "dezastru", "Dezastru",
    "DEZVALUIRI", "dezvaluiri", "Dezvaluiri",
    "DRAM(Ă|A)", "dram(ă|a)", "Dramă",
    "EXCLUSIV", "exclusiv", "Exclusiv",
    "EXCLUSIVITATE", "exclusivitate", "Exclusivitate",
    "EXPLOZIV", "exploziv", "Exploziv",
    "EXTRAORDINAR", "extraordinar", "Extraordinar",
    "FOTO", "foto", "Foto",
    "GALERIE", "galerie", "Galerie",
    "IMAGINI", "imagini", "Imagini",
    "INCENDIAR", "incendiar", "Incendiar",
    "INREGISTRARE", "inregistrare", "Inregistrare",
    "INTERVIU", "interviu", "Interviu",
    "LIVE", "live", "Live",
    "MISTER", "mister", "Mister",
    "PROVOCARE", "provocare", "Provocare",
    "REPLAY", "replay", "Replay",
    "REPORTAJ", "reportaj", "Reportaj",
    "REVELAȚIE", "revelație", "Revelație",
    "SCANDAL", "scandal", "Scandal",
    "SENZAȚIONAL", "senzațional", "Senzațional",
    "ȘOC", "șoc", "ŞOC", "Soc", "soc", "SOC",
    "TRAGEDIE", "tragedie", "Tragedie",
    "UIMITOR", "uimitor", "Uimitor",
    "ULTIM(Ă|A) OR(Ă|A)", "Ultim(ă|a) or(ă|a)", "ultima ora",
    "URGENT", "urgent", "Urgent",
    "VIDEO", "video", "Video"
]

URL_REGEX = r"https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)"
NUMBER_REGEX = r"\d+"
SPACE_REGEX = r"\s+"
SPECIAL_CHARS_REGEX = r"[&#.,?!\"\'‘’“”]|(\([^)]*\))"
BAIT_WORDS_REGEX = r'\b(' + '|'.join(BAIT_WORDS) + r')\b'
EMAIL_REGEX = r'\S*@\S*\s?'
HTML_REGEX = r'<[^<]+?>'

In [ ]:
PT = TypeVar('PT')


class TextPreprocessor(ABC, Generic[PT]):
    def __init__(self) -> None:
        pass

    @abstractmethod
    def __call__(self, dataset: DataFrame) -> PT:
        raise NotImplementedError()


class AutoPreprocessor(TextPreprocessor[BatchEncoding]):
    """Cleans the `text` column of a DataFrame and tokenises it with a Hugging Face tokenizer.

    Returns the cleaned DataFrame and the padded / truncated `input_ids` and `attention_mask` tensors.
    """

    def __init__(self, repo: str, cased: bool = True, max_length: int = 64) -> None:
        super().__init__()

        self.repo: str = repo
        self.cased = cased
        self.tokenizer = AutoTokenizer.from_pretrained(self.repo)
        self.max_length: int = max_length

    def __call__(self, dataset: DataFrame) -> Tuple[DataFrame, BatchEncoding]:
        dataset = dataset.drop(dataset[dataset['text'].str.contains('\t')].index)
        dataset = dataset.reset_index()

        # Regex cleaning
        dataset['text'] = dataset['text'].str.replace(
            URL_REGEX, '', regex=True)
        dataset['text'] = dataset['text'].str.replace(
            NUMBER_REGEX, '', regex=True)
        # Experiments that were not part of the final pipeline:
        # dataset['text'] = dataset['text'].str.replace(
        #     SPECIAL_CHARS_REGEX, '', regex=True)
        # dataset['text'] = dataset['text'].str.replace(
        #     BAIT_WORDS_REGEX, '', regex=True)
        # dataset['text'] = dataset['text'].str.replace(
        #     EMAIL_REGEX, '', regex=True)
        dataset['text'] = dataset['text'].str.replace(
            HTML_REGEX, '', regex=True)
        dataset['text'] = dataset['text'].str.replace(
            SPACE_REGEX, ' ', regex=True)

        if not self.cased:
            dataset['text'] = dataset['text'].str.lower()

        # Character normalisation: cedilla -> comma-below diacritics, typographic -> plain quotes,
        # drop soft hyphens (U+00AD) and en / em dashes
        sentences: List[str] = []
        for text in tqdm(dataset['text'].tolist()):
            text: str = text \
                .replace("ţ", "ț") \
                .replace("ş", "ș") \
                .replace("Ţ", "Ț") \
                .replace("Ş", "Ș") \
                .replace('”', '"') \
                .replace('„', '"') \
                .replace('“', '"') \
                .replace('­', '') \
                .replace('–', '') \
                .replace('—', '')
            sentences.append(text)

        return dataset, self.tokenizer.__call__(
            sentences,
            add_special_tokens=True,
            padding='max_length',
            max_length=self.max_length,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )


class BERTPreprocessor(AutoPreprocessor):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(repo='dumitrescustefan/bert-base-romanian-cased-v1',
                         *args, **kwargs)


class MT5Preprocessor(AutoPreprocessor):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(repo='dumitrescustefan/mt5-base-romanian',
                         *args, **kwargs)


class RoBERTPreprocessor(AutoPreprocessor):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(repo='readerbench/RoBERT-base',
                         cased=False,
                         *args, **kwargs)

## 4. PyTorch dataset and class weights

`SatireDataset` runs a preprocessor once over the whole DataFrame and then serves the `input_ids`, `attention_mask` and, for training data, the `label` of each article.

Balanced accuracy weighs both classes equally no matter how often they occur, so the dataset also computes a weight per class, `1 - class share`. It is passed to the cross-entropy loss, which makes mistakes on the rarer class cost more.

In [ ]:
class SatireDataset(Dataset):
    def __init__(self, dataset: pd.DataFrame, preprocessor: TextPreprocessor[BatchEncoding]):
        super().__init__()

        self.preprocessor: TextPreprocessor[BatchEncoding] = preprocessor
        self.dataset_processor, self.processed_text = self.preprocessor(dataset)

        # Class weights for the loss: the rarer the class, the larger its weight
        self.classes = 'class' in self.dataset_processor.columns
        if self.classes:
            _, class_counts = np.unique(self.dataset_processor['class'], return_counts=True)
            class_counts = torch.from_numpy(class_counts)
            self.weights = 1 - class_counts / class_counts.sum()

    def __len__(self):
        return len(self.dataset_processor)

    def __getitem__(self, idx: int | slice) -> Dict[str, torch.Tensor]:
        input_ids, attention_mask = (self.processed_text['input_ids'][idx], self.processed_text['attention_mask'][idx])

        if self.classes:
            return dict(
                label=torch.tensor(self.dataset_processor['class'].iloc[idx]),
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        else:
            return dict(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

## 5. Models: pretrained encoder + custom classification head

Each model wraps a pretrained Romanian transformer encoder and puts a small fully-connected head on top of the text embedding it produces:

| Model | Encoder | Text embedding | Head |
|---|---|---|---|
| `BERTFlatClassModel` | Romanian BERT (cased) | pooled `[CLS]` vector, 768 | Linear 768→512 → GELU → Linear 512→2, Kaiming init |
| `RoBERTFlatClassModel` | RoBERT-base | pooled `[CLS]` vector, 768 | Dropout 0.1 → Linear 768→2 |
| `MT5FlatClassModel` | Romanian mT5 encoder | all 64 token vectors, flattened (64 × 768 = 49,152) | Linear 49,152→2 |

`PretrainedFlatClassModel.unfreeze()` controls which weights are trained: all of them (`'all'`), none (`'none'`), a list of modules or a list of parameter names. Each model lists the layers to fine-tune: the pooler and the last encoder block for BERT and RoBERT, the last two blocks and the final layer norm for mT5.

> **Note.** Hugging Face loads every pretrained weight as trainable, so `unfreeze([...])` on its own does not freeze anything, and the run reported in section 10 fine-tuned the **whole** encoder. The `freeze_encoder=True` option (added after the hackathon) freezes the encoder first and then unfreezes only the listed layers.

`Ensemble` combines several trained models by summing their logits (soft voting).

In [ ]:
class PretrainedFlatClassModel(ABC, nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

    @abstractmethod
    def create_layers(self) -> nn.Sequential:
        raise NotImplementedError()

    def unfreeze(self, layers: str | List[str] | List[nn.Module], unfreeze: bool = True) -> None:
        """Sets `requires_grad` for 'all' / 'none' of the weights, a list of modules or a list of parameter names."""
        if isinstance(layers, list) and isinstance(layers[0], nn.Module):
            for module in layers:
                for param in module.parameters():
                    param.requires_grad = unfreeze
        elif layers == 'all':
            for param in self.parameters():
                param.requires_grad = unfreeze
        elif layers == 'none':
            self.requires_grad_(False)
            return
        elif isinstance(layers[0], str):
            for (param_name, param) in self.named_parameters():
                if param_name in layers:
                    param.requires_grad = unfreeze
        else:
            raise ValueError('invalid layers param - bad unfreeze')

In [ ]:
class BERTFlatClassModel(PretrainedFlatClassModel):
    def __init__(self,
                 dropout: float = 0.1,
                 freeze_encoder: bool = False,
                 *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.layers = None
        self.dropout: float = dropout
        self.n_classes = 2
        self.repo = 'dumitrescustefan/bert-base-romanian-cased-v1'
        self.bert_model: BertModel = AutoModel.from_pretrained(self.repo)
        if freeze_encoder:
            self.unfreeze([self.bert_model], unfreeze=False)
        self.unfreeze([
            self.bert_model.pooler.dense,
            self.bert_model.encoder.layer[-1:]
        ])
        self.create_layers()

    def create_layers(self) -> None:
        self.layers = nn.Sequential(
            nn.Linear(in_features=768, out_features=512),
            nn.GELU(),
            nn.Linear(in_features=512, out_features=self.n_classes)
        )
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight)

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        input_ids: Tensor = x['input_ids']
        attention_mask: Tensor = x['attention_mask']
        # Second output = pooled [CLS] embedding
        _, output = self.bert_model(
            input_ids, attention_mask, return_dict=False)
        output = self.layers(output)

        return output


class RoBERTFlatClassModel(PretrainedFlatClassModel):
    def __init__(self,
                 dropout: float = 0.1,
                 freeze_encoder: bool = False,
                 *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        # Save internal params
        self.layers = None
        self.dropout: float = dropout

        # Init the base model
        self.n_classes = 2
        self.repo = 'readerbench/RoBERT-base'
        self.bert_model: BertModel = AutoModel.from_pretrained(self.repo)
        if freeze_encoder:
            self.unfreeze([self.bert_model], unfreeze=False)
        self.unfreeze([
            self.bert_model.pooler.dense,
            self.bert_model.encoder.layer[-1:]
        ])
        self.create_layers()

    def create_layers(self) -> None:
        self.layers = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(in_features=768, out_features=self.n_classes)
        )

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        # Extract the relevant data
        input_ids: Tensor = x['input_ids']
        attention_mask: Tensor = x['attention_mask']

        # Call the pretrained model
        _, output = self.bert_model(
            input_ids, attention_mask, return_dict=False)

        # Add layers over it
        output = self.layers(output)

        return output


class MT5FlatClassModel(PretrainedFlatClassModel):
    def __init__(self, freeze_encoder: bool = False, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        # Every token vector goes into the head: 64 tokens x 768 = 49,152 features (needs max_length=64)
        self.fc_head = torch.nn.Linear(49152, 2)
        self.mt5: MT5EncoderModel = MT5EncoderModel.from_pretrained(
            "dumitrescustefan/mt5-base-romanian")
        if freeze_encoder:
            self.unfreeze([self.mt5], unfreeze=False)
        self.unfreeze([
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[11],
            self.mt5.encoder.final_layer_norm
        ])

    def create_layers(self) -> nn.Sequential:
        pass

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        y = self.mt5(
            input_ids=x['input_ids'], attention_mask=x['attention_mask']).last_hidden_state
        y = y.view(y.shape[0], -1)
        y = self.fc_head(y)
        return y

In [ ]:
class Ensemble(nn.Module):
    """Soft voting: sums the logits of several trained models.

    Every member receives the same batch, so the members must share a tokenizer
    (for example the same encoder trained with different seeds).
    """

    def __init__(self, models: List[nn.Module], *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        # Init models (a ModuleList, so that .eval() and .to() reach every member)
        self.models = nn.ModuleList(models)

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        y_hats = []

        for model in self.models:
            y_hat = model.forward(x).detach().cpu()
            y_hats.append(y_hat)

        stacked = torch.stack(y_hats, dim=1)
        predictions = torch.sum(stacked, dim=1)
        return predictions

## 6. Training and evaluation loops

A plain PyTorch loop: AdamW, class-weighted cross-entropy, and the competition metric (balanced accuracy, from scikit-learn) tracked on every batch. `evaluate` runs the model without gradients and also handles the unlabelled test set (`with_labels=False`).

In [ ]:
class Output(object):
    """Predictions plus, for labelled data, the per-batch loss / balanced accuracy and their means."""

    def __init__(self, predictions: List[int], loss_seq: List[float], accy_seq: List[float],
                 with_labels: bool) -> None:
        self.predictions_as_indx: np.ndarray = np.array(predictions)
        self.with_labels: bool = with_labels
        if self.with_labels:
            self.loss_seq: np.ndarray = np.array(loss_seq)
            self.accy_seq: np.ndarray = np.array(accy_seq)
            self.loss_mean: float = self.loss_seq.mean()
            self.accy_mean: float = self.accy_seq.mean()


def train(model: nn.Module, optimizer: Optimizer, loss_fn: nn.Module, train_loader: DataLoader, args: Args, device: torch.device):
    for epoch in range(args.num_epochs):
        model.train()

        epoch_loss: List[float] = []
        epoch_accuracy: List[float] = []
        predictions: List[int] = []
        for batch in tqdm(train_loader):
            batch: Dict[str, torch.Tensor] = {k: v.to(device) for k, v in batch.items()}

            y_true: torch.Tensor | np.ndarray = batch['label']
            y_pred: torch.Tensor | np.ndarray = model.forward(batch)

            optimizer.zero_grad()
            loss: torch.Tensor = loss_fn(y_pred, y_true)
            loss.backward()
            optimizer.step()

            y_true = y_true.detach().cpu().numpy()
            y_pred = y_pred.detach().argmax(dim=1).cpu().numpy()

            predictions.extend(y_pred.tolist())
            epoch_accuracy.append(balanced_accuracy_score(y_true, y_pred))
            epoch_loss.append(loss.detach().cpu().numpy())

        output = Output(predictions, epoch_loss, epoch_accuracy, with_labels=True)
        print('Epoch {}/{} - train loss: {:.4f}, train balanced accuracy: {:.4f}'.format(
            epoch + 1, args.num_epochs, output.loss_mean, output.accy_mean))

    return output


def evaluate(model: nn.Module, loss_fn: nn.Module, data_loader: DataLoader,
             with_labels: bool = True, device: torch.device = torch.device('cpu')) -> Output:
    model.eval()
    with torch.no_grad():
        epoch_loss: List[float] = []
        epoch_accy: List[float] = []
        predictions: List[int] = []

        for _, batch in enumerate(data_loader):
            batch: Dict[str, torch.Tensor] = {k: v.to(device) for k, v in batch.items()}

            y_pred: torch.Tensor | np.ndarray = model.forward(batch)
            if with_labels:
                y_true: torch.Tensor | np.ndarray = batch['label']
                loss: torch.Tensor = loss_fn(y_pred, y_true)

            y_pred = y_pred.detach().argmax(dim=1).cpu().numpy()
            predictions.extend(y_pred.tolist())

            if with_labels:
                y_true = y_true.detach().cpu().numpy()
                epoch_accy.append(balanced_accuracy_score(y_true, y_pred))
                epoch_loss.append(loss.detach().cpu().numpy())

    output = Output(predictions, epoch_loss, epoch_accy, with_labels=with_labels)
    return output

## 7. Experiment setup

`MODEL_NAME` (section 1) picks the encoder; only that model's tokenizer and weights are downloaded. The seeds are fixed right before the model is created, so the head initialisation and the data shuffling are reproducible, which the competition required for every submission.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


MODEL_REGISTRY = {
    'bert': dict(preprocessor=BERTPreprocessor, model=BERTFlatClassModel),
    'RoBERT': dict(preprocessor=RoBERTPreprocessor, model=RoBERTFlatClassModel),
    'mt5': dict(preprocessor=MT5Preprocessor, model=MT5FlatClassModel),
}

args = Args()
setup = MODEL_REGISTRY[MODEL_NAME]

# Clean and tokenise both splits with the selected encoder's tokenizer
preprocessor = setup['preprocessor']()
train_dataset = SatireDataset(train_data_frame, preprocessor)
test_dataset = SatireDataset(test_data_frame, preprocessor)

set_seed(SEED)
model = setup['model']().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
loss_fn = nn.CrossEntropyLoss(weight=train_dataset.weights.to(DEVICE))
train_loader = DataLoader(train_dataset, args.batch_size, shuffle=True)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'{MODEL_NAME}: {n_trainable:,} of {n_total:,} parameters are trainable')

## 8. Training

Three epochs over about 70,000 articles, roughly 6.5 minutes per epoch on a Kaggle GPU.

In [ ]:
train_output: Output = train(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_loader=train_loader,
    args=args,
    device=DEVICE,
)

## 9. Predicting the test set

The test articles go through the same cleaning and tokenisation. Each prediction is the arg-max of the two logits, and the row position of each article is its submission id.

In [ ]:
test_loader: DataLoader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)
test_output: Output = evaluate(
    model=model,
    loss_fn=loss_fn,
    data_loader=test_loader,
    with_labels=False,
    device=DEVICE
)

output: pd.DataFrame = pd.DataFrame({'class': pd.Series(data=test_output.predictions_as_indx)})
output = output.reset_index()
output = output.rename(columns={'index': 'id'})
output.to_csv('submission.csv', index=False)
output.head()

## 10. Results

**Result.** Our team, *Multiplying Tensors*, received an **Honorable Mention** at the hackathon and finished **10th of 45 teams** on the final (private) leaderboard with a balanced accuracy of **0.9375**. On the public leaderboard we scored 0.9389 (9th). The private leaderboard is computed on about 30% of the test set.

**The training run in this notebook** (Romanian BERT, seed 61). The values are averages over the training batches of each epoch:

| Epoch | Train loss | Train balanced accuracy | Time on a Kaggle GPU |
|---|---|---|---|
| 1 | 0.0683 | 0.9745 | 6 min 35 s |
| 2 | 0.0186 | 0.9941 | 6 min 34 s |
| 3 | 0.0090 | 0.9973 | 6 min 34 s |

Balanced accuracy on the training data reaches 0.997, while our submissions scored about 0.94 on the hidden test set. The model fits the training articles much better than it generalises, which is what most of the next steps target.

## 11. What we would do next

- **Validate locally.** The public leaderboard was our only validation signal. A stratified hold-out split or k-fold cross-validation, scored with `compute_metrics`, would give a more reliable estimate than training accuracy, which saturates within the first epoch.
- **Read more of each article.** 64 tokens cover the headline and roughly the first sentence. Longer inputs (256–512 tokens) or head + tail truncation would show the model more of the article.
- **Regularise the fine-tuning.** Freezing most of the encoder (`freeze_encoder=True`), early stopping or smaller learning rates for the lower layers should narrow the gap between training and test accuracy.
- **Ensemble different encoders.** Averaging the predicted probabilities of the BERT, RoBERT and mT5 models, each with its own tokenizer, is a cheap way to gain some accuracy.

### Changes made after the hackathon

- Split the single code cell into documented sections and grouped the imports.
- The training loop logs once per epoch instead of after every batch.
- Only the selected model's tokenizer is loaded and run (the original prepared every model's data up front).
- Fixed the RoBERT setup, which used `BCELoss` (incompatible with 2-class logits) instead of `CrossEntropyLoss`, and registered the `Ensemble` members in an `nn.ModuleList`.
- Added the opt-in `freeze_encoder` flag. With the defaults, the notebook trains the same configuration we ran.
- `DATA_DIR` points at the competition data.

The unmodified notebook exported from Kaggle, with the full training log, is kept in the first commit of this repository.